# Today we begin month 4: Knowledge Graphs, A real Architectural Fix for Mongo DB's Entity-Merging Bug
**Why this is the right way to start Month 4, not just  a new technology to learn**

Tracing back through Month 3's real notebooks aand showing the same big show up, at least four ***separate times***, in four different sessions:

`Entity detected: 'valuation_comparison' (comparison_thread) -> Single-entity active thread fallback resolved: 'valuation_comparison' -> 'total net sales FY2024' 🔁 Match Found! Merging attributes into existing entity: 'total net sales FY2024'`

`Entity detected: 'direct_customer_segments' (strategic_question) -> Single-entity thread fallback resolved: 'direct_customer_segments' -> 'total net sales FY2024' 🔁 Match Found! Merging attributes into existing entity: 'total net sales FY2024'`

A question about market cap valuation and a question about direct customer segments, two completely unrelated topics, both got silently absorbed into an entity called "total net sales FY2024," purely because Month 2 Day 24's `resolve_entity` has a Tier 4 fallback that says "if there's only one existing entity, assume any new mention is about that one." This was flagged as a real risk back when it was built, and Month 3 proved the risk was real, repeatedly.

Here is the actual reason it keeps happening: **MOngo DB**'s `analysis_entities` ***collection has no concept of relationships between distinct things.*** Every Entity is a flat document with an `attributes` dictionary. When the system isn't sure whether a new mention is a new entity or an update to an old one, its only two choices are "create a new flat document" or "merge into the existing flat document" there is no third option, no way to say "These are two separate, related things." A real graph Database makes that third option the *default* representation, not an afterthought, every entity it it's own dostonct node, and relationships between them are explicit, separate edges I have to deliberately create create. Structurally, "everything collapses into one node" isn't even an easy mistake to make with a graph the way it was with a flat collection.

# The Code!: Neo4j as a Genuine Alternative, Tested Against the Actual Bug
**Set Up Neo4j and the Python Driver:** A real, running graph database, eihter Neo4j AuraDB's free tier (cloud, no local install) or Neo4j Desktop locally, plus the `neo4j` Python driver, which speaks Cypher, Neo4j's query language, instead of MongoDB's document-filter syntax.

**A New Extraction Node That Writes Nodes and Relationships, Not Documents:** `extract_graph_entity_node`, structurally similar to Month 2 Day 24's `extract_entities_node`, but instead of building a dictionary to `insert_one`/`update_one`, it writes a real graph node `(MERGE (e: Entity {name:...}))` and a real relationship (`(:User) - [: ASKED_ABOUT] -> (: Entity)`), with each new fact becoming its own separate node connected by an edge, not a key added to a shared attributes blob.

**Replaying the Exact Real Questions That Caused the Bug:** Not synthetic test cases, the actual two questions from Month 3'snreal Day 4 run ("current market capitalization, compared to the valuation implied by FY2024" and "which types of customers does the Company sell directly to?") plus the actual question that was already in the system when they ran ("total net sales FY2024"), fed through the new graph-based extractor in the same order, checking whether they land as three distinct, related nodes instead of one merged blob.

In [7]:
from dataclasses import dataclass
from dotenv import load_dotenv, find_dotenv
from groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
import os
from typing import Optional

Settings.embed_model = HuggingFaceEmbedding(
    model_name='BAAI/bge-m3'
)

load_dotenv(find_dotenv())

client = Groq()

# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


# TOKEN aware LLM Caller

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

In [8]:
# Connecting to a real Graph database

from neo4j import GraphDatabase

# AuraDB free tier: get these three values from your Neo4j Aura console
# after creating a free instance — no local install needed
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with neo4j_driver.session() as session:
    result = session.run("RETURN 'connected' AS status")
    print(f"✅ Neo4j: {result.single()['status']}")

✅ Neo4j: connected


In [3]:
test_user = "month4-day1-replay"

with neo4j_driver.session() as session:
    session.run("MATCH (u:User {id: $uid})-[*0..2]-(n) DETACH DELETE n, u", uid=test_user)
print("✅ Cleared — ready for a clean re-run")

✅ Cleared — ready for a clean re-run


In [4]:
# The graph based Entity Extractor
import re

GRAPH_ENTITY_EXTRACTOR_PROMPT = """You identify a recurring ANALYSIS TOPIC in a conversation
about Apple's FY2024 10-K — a distinct thing being asked about, not a one-off fact.

Respond in EXACTLY this format:
HAS_ENTITY: <YES or NO>
ENTITY_NAME: <a short, specific name for this topic, or NONE>
ENTITY_TYPE: <one of: financial_metric, comparison_thread, strategic_question, or NONE>
"""

def extract_graph_entity(question: str, answer: str) -> Optional[dict]:
    raw = tracked_llm_call(
        messages=[{"role": "user", "content": f"Question: {question}\n\nAnswer: {answer}"}],
        system=GRAPH_ENTITY_EXTRACTOR_PROMPT
    )

    print(f"    [DEBUG] Raw extractor output:\n{raw}\n") # seeing what the model actually said

    has_entity = re.search(r"HAS_ENTITY:\s*(YES|NO)", raw)
    name_match = re.search(r"ENTITY_NAME:\s*(.+)", raw)
    type_match = re.search(r"ENTITY_TYPE:\s*(\w+)", raw)

    if not has_entity or has_entity.group(1) != "YES":
        return None
    name = name_match.group(1).strip() if name_match else None
    if not name or name.upper() == "NONE":
        return None
    return {"name": name, "type": type_match.group(1).strip() if type_match else "financial_metric"}


def write_entity_to_graph(user_id: str, entity_name: str, entity_type: str, question: str):
    """
    Writes a distinct node for THIS entity, and a distinct relationship from
    the user to it. Does NOT check "is there only one existing entity" the
    way Month 2's resolve_entity did — a new topic just becomes a new node,
    connected but never absorbed into an unrelated one.
    """
    with neo4j_driver.session() as session:
        session.run(
            """
            MERGE (u:User {id: $user_id})
            MERGE (e:Entity {name: $entity_name, type: $entity_type})
            MERGE (u)-[r:ASKED_ABOUT]->(e)
            ON CREATE SET r.first_asked = timestamp(), r.times_asked = 1
            ON MATCH SET r.times_asked = r.times_asked + 1
            MERGE (e)-[:RAISED_BY_QUESTION]->(q:Question {text: $question})
            """,
            user_id=user_id, entity_name=entity_name, entity_type=entity_type, question=question
        )
    print(f"  ✅ Graph: '{entity_name}' ({entity_type}) — its own distinct node, related to user '{user_id}'")

In [5]:
# Replaying the Exact Real sequence That caused the Bug

# The exact real sequence from Month 3's actual Day 4 run, in the actual order it happened:
# 1. "total net sales FY2024" already existed as the single active entity
# 2. A question about market cap valuation came in
# 3. A question about direct customer segments came in
# Both #2 and #3 got merged into #1 in the real MongoDB-based system.

test_user = "month4-day1-replay"

write_entity_to_graph(test_user, "total net sales FY2024", "financial_metric",
    "What was Apple's total net sales for fiscal year 2024?")

extracted_2 = extract_graph_entity(
    "What is Apple's current market capitalization as of today, and how does it compare "
    "to the valuation implied by its FY2024 10-K financial statements?",
    "I cannot find sufficient information in the provided context."
)
if extracted_2:
    write_entity_to_graph(test_user, extracted_2["name"], extracted_2["type"],
        "current market capitalization vs FY2024 valuation")

extracted_3 = extract_graph_entity(
    "Which types of customers does the Company sell its products and services directly to?",
    "Small and mid-sized businesses, Education, Enterprise, and Government customers."
)
if extracted_3:
    write_entity_to_graph(test_user, extracted_3["name"], extracted_3["type"],
        "direct customer segments")

  ✅ Graph: 'total net sales FY2024' (financial_metric) — its own distinct node, related to user 'month4-day1-replay'
    [DEBUG] Raw extractor output:
HAS_ENTITY: YES
ENTITY_NAME: Market Capitalization vs FY2024 Valuation
ENTITY_TYPE: comparison_thread

  ✅ Graph: 'Market Capitalization vs FY2024 Valuation' (comparison_thread) — its own distinct node, related to user 'month4-day1-replay'
    [DEBUG] Raw extractor output:
HAS_ENTITY: YES
ENTITY_NAME: Customer Segments
ENTITY_TYPE: strategic_question

  ✅ Graph: 'Customer Segments' (strategic_question) — its own distinct node, related to user 'month4-day1-replay'


In [6]:
# Querying the Graph back, Confirming three distinct nodes, Not one

with neo4j_driver.session() as session:
    result = session.run(
        """
        MATCH (u:User {id: $user_id})-[r:ASKED_ABOUT]->(e:Entity)
        RETURN e.name AS entity, e.type AS type, r.times_asked AS times_asked
        ORDER BY r.first_asked
        """,
        user_id=test_user
    )
    rows = list(result)

print(f"\n{'='*60}\nENTITIES FOR '{test_user}', should be 3 DISTINCT nodes\n{'='*60}")
for row in rows:
    print(f"  {row['entity']:<40} ({row['type']}), asked {row['times_asked']}x")

print(f"\n{'✅ Bug avoided' if len(rows) == 3 else '⚠️ Still collapsed —> investigate'}: "
      f"{len(rows)} distinct entities found (expected 3)")
print(f"\nCompare against the real MongoDB result: all three of these collapsed into")
print(f"ONE entity ('total net sales FY2024') via the single-entity fallback.")


ENTITIES FOR 'month4-day1-replay', should be 3 DISTINCT nodes
  total net sales FY2024                   (financial_metric), asked 1x
  Market Capitalization vs FY2024 Valuation (comparison_thread), asked 1x
  Customer Segments                        (strategic_question), asked 1x

✅ Bug avoided: 3 distinct entities found (expected 3)

Compare against the real MongoDB result: all three of these collapsed into
ONE entity ('total net sales FY2024') via the single-entity fallback.
